# Authentication & Access Audit

This notebook queries the OCP audit SQLite datastore to present findings for:

- **OCP-1 — OAuth External Auth**: Identity provider configuration, external auth enforcement, kubeadmin removal
- **OCP-2 — Granular RBAC**: ClusterRoles, ClusterRoleBindings, cluster-admin holders, wildcard permissions, self-provisioner status
- **OCP-3 — API & Console Access Restriction**: Access to the cluster API and Console is restricted to required admins (TLS profile, audit profile, client CA, encryption, CORS allow-list, cluster-admin subject count)
- **OCP-4 — Worker Node AuthN/AuthZ**: Worker node authentication and authorization enforcement (kubelet anonymous auth, authorization mode, MachineConfig drift)
- **OCP-5 — Cluster Admin/SRE Credential Management**: Separation and protection of cluster-admin and infrastructure credentials (kubeadmin removal, critical-namespace secret inventory, cluster-admin binding subjects)


In [ ]:
import os
import sys

import pandas as pd

# Shared notebook helpers (sys.path + OCP_AUDIT_DB + styling)
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from notebook_style import bootstrap, style_table  # noqa: E402

print("python:", sys.executable)
print("cwd:", os.getcwd())

session, engine = bootstrap()

from sqlalchemy import func  # noqa: E402
from schema.models import (  # noqa: E402
    ApiServerConsoleAccess,
    Cluster,
    ClusterAdminBinding,
    ClusterRole,
    ClusterRoleBinding,
    ClusterRoleBindingSubject,
    ClusterRoleRule,
    ClusterRoleRuleNonResourceUrl,
    ClusterRoleRuleResource,
    ClusterRoleRuleVerb,
    CredentialManagementSecret,
    OAuthExternalAuth,
    SelfProvisionerBinding,
    SelfProvisionerSubject,
    WorkerNodeAuth,
)

print(f"Connected to: {engine.url}")


## Cluster Inventory

In [ ]:
df_clusters = pd.read_sql(
    session.query(
        Cluster.id,
        Cluster.cluster_name,
        Cluster.cluster_context,
        Cluster.cluster_server,
    ).statement,
    engine,
)
print(f"{len(df_clusters)} cluster(s) in dataset")
style_table(df_clusters)

---
## OCP-1: Authentication Posture

Per-cluster summary of external authentication enforcement and kubeadmin account status.

In [ ]:
df_auth = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        OAuthExternalAuth.external_auth_enforced,
        OAuthExternalAuth.kubeadmin_removed,
        OAuthExternalAuth.identity_providers_count,
    )
    .join(Cluster, OAuthExternalAuth.cluster_id == Cluster.id)
    .distinct()
    .statement,
    engine,
)
print("Authentication posture by cluster")
style_table(df_auth)

### OCP-1: Identity Provider Inventory

All configured identity providers across clusters.

In [ ]:
df_idp = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        OAuthExternalAuth.idp_name,
        OAuthExternalAuth.idp_type,
        OAuthExternalAuth.idp_mapping_method,
        OAuthExternalAuth.idp_issuer,
        OAuthExternalAuth.access_token_max_age_seconds,
    )
    .join(Cluster, OAuthExternalAuth.cluster_id == Cluster.id)
    .statement,
    engine,
)
print(f"{len(df_idp)} identity provider(s) configured")
style_table(df_idp)

### OCP-1: Compliance Flags

Clusters where external auth is **not** enforced or kubeadmin has **not** been removed.

In [ ]:
df_flags = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        OAuthExternalAuth.external_auth_enforced,
        OAuthExternalAuth.kubeadmin_removed,
        OAuthExternalAuth.idp_name,
        OAuthExternalAuth.idp_type,
    )
    .join(Cluster, OAuthExternalAuth.cluster_id == Cluster.id)
    .filter(
        (OAuthExternalAuth.external_auth_enforced == False)  # noqa: E712
        | (OAuthExternalAuth.kubeadmin_removed == False)  # noqa: E712
    )
    .statement,
    engine,
)
if df_flags.empty:
    print("All clusters compliant — no findings.")
else:
    print(f"{len(df_flags)} non-compliant finding(s)")
style_table(df_flags)

---
## OCP-2: Cluster-Admin Holders

All subjects bound to the `cluster-admin` ClusterRole — these have full administrative privileges.

In [ ]:
df_admins = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterRoleBinding.binding_name,
        ClusterRoleBindingSubject.subject_kind,
        ClusterRoleBindingSubject.subject_name,
        ClusterRoleBindingSubject.subject_namespace,
    )
    .join(Cluster, ClusterRoleBinding.cluster_id == Cluster.id)
    .join(
        ClusterRoleBindingSubject,
        ClusterRoleBindingSubject.clusterrolebinding_id == ClusterRoleBinding.id,
    )
    .filter(ClusterRoleBinding.role_ref_name == "cluster-admin")
    .order_by(Cluster.cluster_name, ClusterRoleBindingSubject.subject_kind)
    .statement,
    engine,
)
print(f"{len(df_admins)} cluster-admin binding(s) across all clusters")
style_table(df_admins)

### OCP-2: Wildcard Permissions

ClusterRoles containing rules with wildcard (`*`) verbs or resources — these grant broad access.

In [ ]:
# Roles with wildcard verbs
wild_verbs = (
    session.query(
        Cluster.cluster_name,
        ClusterRole.role_name,
        func.group_concat(ClusterRoleRuleVerb.verb, "; ").label("verbs"),
    )
    .join(ClusterRole, ClusterRole.cluster_id == Cluster.id)
    .join(ClusterRoleRule, ClusterRoleRule.clusterrole_id == ClusterRole.id)
    .join(ClusterRoleRuleVerb, ClusterRoleRuleVerb.rule_id == ClusterRoleRule.id)
    .filter(ClusterRoleRuleVerb.verb == "*")
    .group_by(Cluster.cluster_name, ClusterRole.role_name)
)

# Roles with wildcard resources
wild_res = (
    session.query(
        Cluster.cluster_name,
        ClusterRole.role_name,
        func.group_concat(ClusterRoleRuleResource.resource, "; ").label("resources"),
    )
    .join(ClusterRole, ClusterRole.cluster_id == Cluster.id)
    .join(ClusterRoleRule, ClusterRoleRule.clusterrole_id == ClusterRole.id)
    .join(
        ClusterRoleRuleResource,
        ClusterRoleRuleResource.rule_id == ClusterRoleRule.id,
    )
    .filter(ClusterRoleRuleResource.resource == "*")
    .group_by(Cluster.cluster_name, ClusterRole.role_name)
)

df_wild_verbs = pd.read_sql(wild_verbs.statement, engine)
df_wild_res = pd.read_sql(wild_res.statement, engine)

df_wildcards = pd.merge(
    df_wild_verbs, df_wild_res, on=["cluster_name", "role_name"], how="outer"
).fillna("")

print(f"{len(df_wildcards)} role(s) with wildcard permissions")
style_table(df_wildcards)

### OCP-2: Non-Resource URL Access

ClusterRoles granting access to non-resource URLs (API discovery, health endpoints).

In [ ]:
df_nru = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterRole.role_name,
        func.group_concat(
            ClusterRoleRuleNonResourceUrl.non_resource_url, "; "
        ).label("non_resource_urls"),
    )
    .join(ClusterRole, ClusterRole.cluster_id == Cluster.id)
    .join(ClusterRoleRule, ClusterRoleRule.clusterrole_id == ClusterRole.id)
    .join(
        ClusterRoleRuleNonResourceUrl,
        ClusterRoleRuleNonResourceUrl.rule_id == ClusterRoleRule.id,
    )
    .group_by(Cluster.cluster_name, ClusterRole.role_name)
    .statement,
    engine,
)
print(f"{len(df_nru)} role(s) with non-resource URL permissions")
style_table(df_nru)

### OCP-2: Self-Provisioner Status

Self-provisioner bindings control whether users can create their own projects. Clusters **missing** from this table have self-provisioning disabled.

In [ ]:
df_sp = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        SelfProvisionerBinding.binding_name,
        SelfProvisionerBinding.role_ref_name,
        SelfProvisionerSubject.subject_kind,
        SelfProvisionerSubject.subject_name,
    )
    .join(Cluster, SelfProvisionerBinding.cluster_id == Cluster.id)
    .join(
        SelfProvisionerSubject,
        SelfProvisionerSubject.binding_id == SelfProvisionerBinding.id,
    )
    .order_by(Cluster.cluster_name)
    .statement,
    engine,
)

all_clusters = {c.cluster_name for c in session.query(Cluster).all()}
sp_clusters = set(df_sp["cluster_name"].unique()) if not df_sp.empty else set()
disabled = all_clusters - sp_clusters

print(f"{len(df_sp)} self-provisioner binding(s)")
if disabled:
    print(f"Self-provisioning DISABLED on: {', '.join(sorted(disabled))}")
style_table(df_sp)

---
### OCP-2: RBAC Summary by Cluster

Aggregated counts per cluster for a high-level view of RBAC scope.

In [ ]:
role_counts = (
    session.query(
        Cluster.cluster_name,
        func.count(ClusterRole.id).label("roles"),
    )
    .join(ClusterRole, ClusterRole.cluster_id == Cluster.id)
    .group_by(Cluster.cluster_name)
)

binding_counts = (
    session.query(
        Cluster.cluster_name,
        func.count(ClusterRoleBinding.id).label("bindings"),
    )
    .join(ClusterRoleBinding, ClusterRoleBinding.cluster_id == Cluster.id)
    .group_by(Cluster.cluster_name)
)

admin_counts = (
    session.query(
        Cluster.cluster_name,
        func.count(ClusterRoleBindingSubject.id).label("cluster_admin_subjects"),
    )
    .join(ClusterRoleBinding, ClusterRoleBinding.cluster_id == Cluster.id)
    .join(
        ClusterRoleBindingSubject,
        ClusterRoleBindingSubject.clusterrolebinding_id == ClusterRoleBinding.id,
    )
    .filter(ClusterRoleBinding.role_ref_name == "cluster-admin")
    .group_by(Cluster.cluster_name)
)

df_roles = pd.read_sql(role_counts.statement, engine)
df_bindings = pd.read_sql(binding_counts.statement, engine)
df_admin_c = pd.read_sql(admin_counts.statement, engine)

df_summary = df_roles.merge(df_bindings, on="cluster_name", how="outer").merge(
    df_admin_c, on="cluster_name", how="outer"
).fillna(0)

for col in ["roles", "bindings", "cluster_admin_subjects"]:
    df_summary[col] = df_summary[col].astype(int)

print("RBAC summary by cluster")
style_table(df_summary)

---
## OCP-3: API & Console Access Restriction

Access to the cluster API server and Web Console must be restricted to required administrators. This section reports the per-cluster hardening posture: TLS profile, audit profile, encryption-at-rest, client CA, CORS allow-list, custom serving certificates, and the number of subjects holding `cluster-admin`.

In [ ]:
df_api_posture = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ApiServerConsoleAccess.api_server_url,
        ApiServerConsoleAccess.console_url,
        ApiServerConsoleAccess.tls_security_profile_type,
        ApiServerConsoleAccess.tls_min_version,
        ApiServerConsoleAccess.audit_profile,
        ApiServerConsoleAccess.client_ca_name,
        ApiServerConsoleAccess.encryption_type,
        ApiServerConsoleAccess.serving_certs_count,
        ApiServerConsoleAccess.cluster_admin_binding_count,
    )
    .join(Cluster, ApiServerConsoleAccess.cluster_id == Cluster.id)
    .order_by(Cluster.cluster_name)
    .statement,
    engine,
)
print(f"API / Console access posture for {len(df_api_posture)} cluster(s)")
style_table(df_api_posture)


### OCP-3: Compliance Flags

Clusters that fail one or more hardening checks for API / Console access:

- **TLS profile** not `Intermediate` or `Modern`, or `tls_min_version` below `VersionTLS12`
- **Audit profile** missing or `Default` (should be `WriteRequestBodies` or stricter)
- **Encryption-at-rest** not enabled (empty or `identity`)
- **cluster-admin subjects** greater than a reasonable threshold (default: **3**)

In [ ]:
ADMIN_THRESHOLD = 3

df_api = df_api_posture.copy()

def _s(val):
    """Safe string: treat NaN/None as empty."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return ""
    return str(val).strip()

def _tls_weak(row):
    prof = _s(row.get("tls_security_profile_type"))
    ver = _s(row.get("tls_min_version"))
    if prof not in ("Intermediate", "Modern"):
        return True
    if ver and ver < "VersionTLS12":
        return True
    return False

def _audit_weak(row):
    return _s(row.get("audit_profile")) in ("", "Default", "None")

def _encryption_weak(row):
    return _s(row.get("encryption_type")).lower() in ("", "identity")

def _too_many_admins(row):
    n = row.get("cluster_admin_binding_count")
    if n is None or (isinstance(n, float) and pd.isna(n)):
        return False
    return n > ADMIN_THRESHOLD

df_api["tls_weak"] = df_api.apply(_tls_weak, axis=1)
df_api["audit_weak"] = df_api.apply(_audit_weak, axis=1)
df_api["encryption_weak"] = df_api.apply(_encryption_weak, axis=1)
df_api["too_many_admins"] = df_api.apply(_too_many_admins, axis=1)

flag_cols = ["tls_weak", "audit_weak", "encryption_weak", "too_many_admins"]
df_api_flags = df_api[df_api[flag_cols].any(axis=1)][
    [
        "cluster_name",
        "tls_security_profile_type",
        "tls_min_version",
        "audit_profile",
        "encryption_type",
        "cluster_admin_binding_count",
        *flag_cols,
    ]
]

print(f"{len(df_api_flags)} cluster(s) fail at least one OCP-3 hardening check")
style_table(df_api_flags)


---
## OCP-4: Worker Node AuthN/AuthZ

Worker node authentication and authorization must be enforced on every node. This section shows per-node kubelet posture (version, ready status, anonymous-auth, authorization mode) and MachineConfig drift (whether each node's `currentConfig` matches its `desiredConfig`).

In [ ]:
df_nodes = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        WorkerNodeAuth.node_name,
        WorkerNodeAuth.node_roles,
        WorkerNodeAuth.kubelet_version,
        WorkerNodeAuth.ready_status,
        WorkerNodeAuth.machine_config_state,
        WorkerNodeAuth.configs_match,
        WorkerNodeAuth.kubelet_config_count,
        WorkerNodeAuth.anonymous_auth,
        WorkerNodeAuth.authorization_mode,
    )
    .join(Cluster, WorkerNodeAuth.cluster_id == Cluster.id)
    .order_by(Cluster.cluster_name, WorkerNodeAuth.node_name)
    .statement,
    engine,
)
print(f"Worker-node auth posture: {len(df_nodes)} node(s)")
style_table(df_nodes)


### OCP-4: Compliance Flags

Nodes that fail one or more worker-node authN/authZ checks:

- **Anonymous auth** enabled (kubelet accepts unauthenticated requests)
- **Authorization mode** not `Webhook` (expected default) or `RBAC`
- **Not Ready** — node is not healthy
- **MachineConfig drift** — `currentConfig` does not match `desiredConfig`

In [ ]:
df_n = df_nodes.copy()

def _anon_weak(val):
    s = ("" if val is None else str(val)).strip().lower()
    return s in ("true", "1", "yes")

def _authz_weak(val):
    s = ("" if val is None else str(val)).strip()
    if s in ("", "default"):
        return False  # default == Webhook in OCP 4.x
    return s not in ("Webhook", "RBAC")

def _not_ready(val):
    return str(val or "").strip() != "True"

def _drift(val):
    if val is None:
        return False
    return not bool(val)

df_n["anonymous_auth_enabled"] = df_n["anonymous_auth"].apply(_anon_weak)
df_n["authz_mode_weak"] = df_n["authorization_mode"].apply(_authz_weak)
df_n["not_ready"] = df_n["ready_status"].apply(_not_ready)
df_n["mc_drift"] = df_n["configs_match"].apply(_drift)

flag_cols = ["anonymous_auth_enabled", "authz_mode_weak", "not_ready", "mc_drift"]
df_node_flags = df_n[df_n[flag_cols].any(axis=1)][
    [
        "cluster_name",
        "node_name",
        "node_roles",
        "ready_status",
        "anonymous_auth",
        "authorization_mode",
        "configs_match",
        *flag_cols,
    ]
]

print(f"{len(df_node_flags)} node(s) fail at least one OCP-4 hardening check")
style_table(df_node_flags)


---
## OCP-5: Cluster Admin / SRE Credential Management

Cluster-admin and infrastructure credentials must be segregated and highly protected. This section reports the kubeadmin-break-glass account status, an inventory of secrets in the critical admin namespaces (`kube-system`, `openshift-config`, `openshift-config-managed`), and every subject currently holding the `cluster-admin` role.

In [ ]:
# Per-cluster summary: kubeadmin status + secret counts in critical namespaces
df_cred_summary = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        func.max(CredentialManagementSecret.kubeadmin_exists).label("kubeadmin_exists"),
        func.count(CredentialManagementSecret.id).label("critical_secret_count"),
    )
    .join(Cluster, CredentialManagementSecret.cluster_id == Cluster.id)
    .group_by(Cluster.cluster_name)
    .order_by(Cluster.cluster_name)
    .statement,
    engine,
)
print(f"Credential management summary: {len(df_cred_summary)} cluster(s)")
style_table(df_cred_summary)


### OCP-5: Critical-Namespace Secret Inventory

Full inventory of secrets living in the critical admin namespaces, with age in days. Long-lived Opaque secrets and `kubernetes.io/service-account-token` entries are candidates for rotation review.

In [ ]:
df_secrets = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        CredentialManagementSecret.namespace,
        CredentialManagementSecret.secret_name,
        CredentialManagementSecret.secret_type,
        CredentialManagementSecret.age_days,
        CredentialManagementSecret.service_account,
        CredentialManagementSecret.creation_timestamp,
    )
    .join(Cluster, CredentialManagementSecret.cluster_id == Cluster.id)
    .order_by(
        Cluster.cluster_name,
        CredentialManagementSecret.namespace,
        CredentialManagementSecret.secret_name,
    )
    .statement,
    engine,
)
print(f"Critical-namespace secrets: {len(df_secrets)} row(s)")
style_table(df_secrets)


### OCP-5: Cluster-Admin Binding Subjects

Every subject (User, Group, or ServiceAccount) currently bound to the `cluster-admin` role — these identities have full administrative control and must be tightly scoped.

In [ ]:
df_admin_bindings = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterAdminBinding.binding_name,
        ClusterAdminBinding.role_ref_name,
        ClusterAdminBinding.subject_kind,
        ClusterAdminBinding.subject_name,
        ClusterAdminBinding.subject_namespace,
        ClusterAdminBinding.creation_timestamp,
    )
    .join(Cluster, ClusterAdminBinding.cluster_id == Cluster.id)
    .order_by(
        Cluster.cluster_name,
        ClusterAdminBinding.binding_name,
        ClusterAdminBinding.subject_name,
    )
    .statement,
    engine,
)
print(f"Cluster-admin binding subjects: {len(df_admin_bindings)} row(s)")
style_table(df_admin_bindings)


### OCP-5: Compliance Flags

Clusters that fail one or more credential-management hardening checks:

- **kubeadmin still present** — the break-glass account has not been removed
- **Aged Opaque secrets** — any secret in a critical namespace older than the rotation threshold (default: **365 days**)

In [ ]:
ROTATION_DAYS = 365

df_kubeadmin = df_cred_summary[df_cred_summary["kubeadmin_exists"] == 1][
    ["cluster_name", "kubeadmin_exists"]
].copy()
df_kubeadmin["kubeadmin_still_present"] = True

df_aged = df_secrets[
    (df_secrets["age_days"].fillna(0) > ROTATION_DAYS)
][["cluster_name", "namespace", "secret_name", "secret_type", "age_days"]].copy()

print(
    f"{len(df_kubeadmin)} cluster(s) still have the kubeadmin account; "
    f"{len(df_aged)} secret(s) older than {ROTATION_DAYS} days in critical namespaces."
)
if len(df_kubeadmin):
    style_table(df_kubeadmin)
if len(df_aged):
    style_table(df_aged)


In [ ]:
session.close()
print("Session closed.")